## Data Sampling

Training of LLMs is done in an autoregressive way meaning that the model takes a set of words or tokens as input and outputs the estimated next word.

This process is called self-supervised learning because of its natural implementation way. We don't have to label the outputs manually. It will be automatically done and used in training process.

Here, there is an important parameter called `context_length` or `context_size` which defines the length of the set of input tokens to be used for the next word prediction at the same time. For example, if the `context_size` is selected as 4, then the model will predict the 5th token by using preceding 4 tokens. Then, it will slide by the `window_size` and continue for the next token predictions. 

In order to create the training data using the token, we need to go through the following steps:
1. Import the raw text data and tokenize the words or sub-words
2. Create a tokenizer (we can use BPE tokenizer from tiktoken)
3. Use a for loop to select `n-1` tokens as the training input and `n`th element as training output
    * In this step we need to define two parameters: `context_size` and `window_size`
4. Check the implementation using a test-case.

### Step 1: Import the text to be used for the generation of training data

In [29]:
from pathlib import Path


f_path = Path("../data/the-verdict.txt")
with open(f_path, 'r', encoding="utf-8") as f:
    raw_text = f.read()

print(len(raw_text))
print(raw_text[:50])

20479
I HAD always thought Jack Gisburn rather a cheap g


### Step 2: Create the tokenizer

In [40]:
import tiktoken


tokenizer = tiktoken.get_encoding("gpt2")

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

sample_length = 29
enc_sample = enc_text[:sample_length]
print(enc_sample)

5145
[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438, 568, 340, 373, 645, 1049, 5975, 284, 502, 284]


### Step 3: Implement Data Sampler

In [ ]:
from typing import List


class DataSamplerV1:

    def __init__(self, token_list: List[int], context_size: int, window_size: int):
        self.token_list = token_list
        self.context_size = context_size
        self.window_size = window_size

        self.features = []
        self.labels = []

    def create_dataset(self):
        for idx in range(0, len(self.token_list) - self.context_size, self.window_size):
            self.features.append(self.token_list[idx : idx + self.context_size])
            self.labels.append(self.token_list[idx + 1 : idx + self.context_size + 1])

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

sampler = DataSamplerV1(enc_sample, context_size=4, window_size=4)
sampler.create_dataset()
X, y = sampler[-2:]
for k, v in zip(X, y):
    print(f"Token: {k} -> {v}")
    print(f"Text: {tokenizer.decode(k)} -> {tokenizer.decode(v)}")
    print("")

print("Total length of dataset: ", len(sampler))

Token: [568, 340, 373, 645] -> [340, 373, 645, 1049]
Text: so it was no ->  it was no great

Token: [1049, 5975, 284, 502] -> [5975, 284, 502, 284]
Text:  great surprise to me ->  surprise to me to

Total length of dataset:  7


So far, we have implemented a basic data sampler class, `DataSamplerV1`. Now, we can turn the sampler class a torch dataset and implement a dataloader to use the dataset.

Hence, it will support torch tensors and be ready for the training.

In [47]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):

    def __init__(self, text: str, tokenizer: tiktoken.Encoding, context_size: int, window_size: int):
        self.context_size = context_size
        self.window_size = window_size
        self.token_ids = tokenizer.encode(text)
        self.inputs = []
        self.targets = []
        self.create_dataset()

    def create_dataset(self):
        for idx in range(0, len(self.token_ids) - self.context_size, self.window_size):
            input_chunk = self.token_ids[idx : idx + self.context_size]
            target_chunk = self.token_ids[idx + 1 : idx + self.context_size + 1]
            self.inputs.append(torch.tensor(input_chunk))
            self.targets.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        return self.inputs[index], self.targets[index]
    

def gpt_loader_v1(text: str, batch_size: int, context_size=256, window_size=128, shuffle=True, num_workers=0, drop_last=True):
    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(
        text=text,
        tokenizer=tokenizer,
        context_size=context_size,
        window_size=window_size
    )

    loader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        drop_last=drop_last
    )

    return loader


f_path = Path("../data/the-verdict.txt")
with open(f_path, 'r', encoding="utf-8") as f:
    raw_text = f.read()

dataloader = gpt_loader_v1(raw_text, batch_size=1, context_size=4, window_size=1, shuffle=False)
data_iter = iter(dataloader)
print("First batch:")
print(next(data_iter))
print("")

dataloader = gpt_loader_v1(
    raw_text, batch_size=8, context_size=4, window_size=4, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

First batch:
[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
